# CounTR sobre MinneApple (zero-shot y 3-shot) · CINF200 Entrega 2
Entorno: Colab con GPU (Entorno de ejecución → Cambiar tipo → T4). Tiempo estimado: ~15 min.

In [ ]:
!nvidia-smi -L
!git clone -q https://github.com/Verg-Avesta/CounTR.git
%cd CounTR
!pip -q install timm==0.4.9 gdown pycocotools
# Parches de compatibilidad con PyTorch 2.x / NumPy 2.x de Colab actual
!grep -rl 'torch._six' /usr/local/lib/python3*/dist-packages/timm | xargs -r sed -i 's/from torch._six import container_abcs/import collections.abc as container_abcs/'
!grep -rl 'torch._six' . | xargs -r sed -i 's/from torch._six import inf/from math import inf/'
!grep -rl 'np.float)' . | xargs -r sed -i 's/np.float)/np.float64)/g'

In [ ]:
# Pesos oficiales ajustados en FSC-147 (enlace del README del repositorio)
!mkdir -p weights && gdown -q 1CzYyiYqLshMdqJ9ZPFJyIzXBa7uFUIYZ -O weights/FSC147.pth
!ls -la weights

In [ ]:
# MISMO test que la U-Net (splits.json): 2 videos de train sorteados con semilla 2026, 125 imagenes.
# (Las anotaciones del test oficial en este espejo estan desalineadas; ver informe 2.2.)
import os, json, urllib.request
from concurrent.futures import ThreadPoolExecutor
R='https://huggingface.co/datasets/lauesa1/minne-apple-segmentation/resolve/main/train/'
TEST_VIDEOS=('20150919_174151','20150921_131234')
os.makedirs('/content/mapple/images', exist_ok=True)
urllib.request.urlretrieve(R+'train.json', '/content/mapple/train.json')
coco=json.load(open('/content/mapple/train.json'))
coco['images']=[im for im in coco['images'] if im['file_name'].startswith(TEST_VIDEOS)]
def get(im):
    p='/content/mapple/images/'+im['file_name']
    if not os.path.exists(p): urllib.request.urlretrieve(R+'images/'+im['file_name'], p)
with ThreadPoolExecutor(16) as ex: list(ex.map(get, coco['images']))
print(len(coco['images']), 'imagenes de test (esperado: 125)')

In [ ]:
import numpy as np, torch, torch.nn as nn, pandas as pd, time
from PIL import Image
from torchvision import transforms
import models_mae_cross
dev=torch.device('cuda')
model=models_mae_cross.__dict__['mae_vit_base_patch16'](norm_pix_loss='store_true').to(dev)
ck=torch.load('weights/FSC147.pth', map_location='cpu', weights_only=False)
print(model.load_state_dict(ck['model'], strict=False))
model.eval();

In [ ]:
# Las fotos son verticales (720x1280) y las manzanas pequeñas: se divide en mosaicos
# cuadrados de TILE px (original) reescalados a 384x384, que es la entrada nativa de CounTR.
# Ejemplares (3-shot): 3 cajas anotadas elegidas al azar (semilla fija) por imagen,
# recortadas a la escala del mosaico y redimensionadas a 64x64 (protocolo FSC-147).
TILE=360
to_t=transforms.ToTensor()
@torch.no_grad()
def count(img, ex_boxes, shots):
    W,H=img.size; s=384/TILE; total=0.0
    big=to_t(img.resize((round(W*s), round(H*s))))
    if shots:
        ex=[transforms.Resize((64,64))(big[:, int(y*s):int((y+h)*s)+1, int(x*s):int((x+w)*s)+1]) for x,y,w,h in ex_boxes]
        boxes=torch.stack(ex)[None].to(dev)
    else:
        boxes=torch.zeros(1,0,3,64,64, device=dev)
    for y0 in range(0, H, TILE):
        for x0 in range(0, W, TILE):
            t=img.crop((x0, y0, x0+TILE, y0+TILE))  # rellena con negro fuera de la imagen
            x=to_t(t.resize((384,384)))[None].to(dev)
            out,=model(x, boxes, shots)
            total+=out.sum().item()/60
    return total
rng=np.random.default_rng(2026)
anns={}
for a in coco['annotations']: anns.setdefault(a['image_id'],[]).append(a['bbox'])
rows=[]; t0=time.time()
for i,im in enumerate(coco['images']):
    img=Image.open('/content/mapple/images/'+im['file_name']).convert('RGB')
    bb=anns.get(im['id'],[])
    ex=[bb[j] for j in rng.choice(len(bb), min(3,len(bb)), replace=False)] if bb else []
    rows.append({'stem':im['file_name'][:-4], 'gt':len(bb),
                 'countr_0shot':count(img, [], 0),
                 'countr_3shot':count(img, ex, len(ex)) if len(ex)==3 else float('nan')})
    if i%25==0: print(i, rows[-1], f'{time.time()-t0:.0f}s')
df=pd.DataFrame(rows); df.to_csv('/content/countr_test_predictions.csv', index=False)
for c in ['countr_0shot','countr_3shot']:
    e=(df[c]-df['gt']).dropna(); print(c, 'MAE', e.abs().mean().round(2), 'RMSE', np.sqrt((e**2).mean()).round(2),
          'R2', (1-(e**2).sum()/((df['gt']-df['gt'].mean())**2).sum()).round(3))

In [ ]:
from google.colab import files; files.download('/content/countr_test_predictions.csv')